In [0]:
%python
!pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
%python

from pyspark.sql.functions import col
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import pandas as pd
from pyspark.ml import Pipeline
from databricks.feature_store.client import FeatureStoreClient
import mlflow
from mlflow.models.signature import infer_signature


# Initialize the client
fe = FeatureEngineeringClient()


# basic info to collect
source_table = 'ai2605.ai.petstreaming_sales'
feature_table = 'ai2605.ai.petstreaming_sales_features'
model_name = 'ai2605.ai.petstreaming_sales_model'

spark_df = spark.table(source_table)
#display(spark_df)
df_prepared = spark_df.withColumn('label', col('is_subscription').cast('double'))
#display(df_prepared)
# Step 1: Create Feature Table
features_df = df_prepared.select('transaction_id', 'units_sold', 'price', 'category' )
#display(features_df)

fe.create_table(
    name=feature_table,
    primary_keys='transaction_id',
    df=features_df,
    description='features for petstreaming sales',
    schema=features_df.schema,
)

# Step 2: Create Training Set
label_df = df_prepared.select('transaction_id', 'label')

feature_lookups = [
    FeatureLookup(
        table_name=feature_table,
        lookup_key='transaction_id',
        feature_names=['units_sold', 'price', 'category'])]


training_set = fe.create_training_set(
    df = label_df,
    label='label',
    exclude_columns=['transaction_id'],
    feature_lookups=feature_lookups
)
training_Df = training_set.load_df()
display(training_Df)

from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
# 3. PIPELINE DEFINITION
indexer = StringIndexer(inputCol="category", outputCol="category_idx")
encoder = OneHotEncoder(inputCol="category_idx", outputCol="category_vec")
assembler = VectorAssembler(inputCols=["units_sold", "price", "category_vec"], outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(training_Df)

#[ Step 3: Run Original Pipeline ]  

input_example = pd.DataFrame({
    "units_sold": [40],
    "price": [32.79],
    "category": ['Toys']})

output_example = pd.DataFrame({"prediction": [0.0]})
signature = infer_signature(input_example, output_example)

# 6. LOG AND REGISTER
model_name = "ai2605.ai.petstreaming_sales_model_with_fs"
volume_path = "/Volumes/ai2605/ai/ml_temp"
with mlflow.start_run(run_name="production_v1"):
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        signature=signature,
        registered_model_name=model_name,
        dfs_tmpdir=volume_path
    )

print(f"Model successfully logged and registered as: {model_name}")

#   ▼
#[ Step 4: fe.log_model() ] ──> [ Registered Model with FS Metadata ] 